# Load package

In [ ]:
import pandas as pd
import scimap as sm
import numpy as np

# Load data

In [ ]:
data = pd.read_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_2.csv', index_col=0)

# Spatial Count

In [ ]:
image_path = '/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_2.csv'
# Convert the data to scimap format
adata = sm.pp.mcmicro_to_scimap(image_path, 
                                remove_dna=False, remove_string_from_name=None, log=False, drop_markers=None,
                                random_sample=None, unique_CellId=True, CellId='CellID', split='X_centroid',
                                custom_imageid=None, min_cells=None, output_dir=None)
adata.obsm['spatial'] = np.array(adata.obs[['Y_centroid', 'X_centroid']])

In [ ]:
adata = sm.tl.spatial_count(adata, phenotype='celltype_1', method='radius', radius=45, label='spatial_count_45px')

In [ ]:
adata = sm.tl.spatial_cluster(adata, df_name='spatial_count_45px', method='kmeans', k=10, label='SC45px_neigh_kmeans_10')

In [7]:
# create a mapping DataFrame from adata.obs with the columns we need
mapping_df = adata.obs[['CellID', 'core_imageid', 'SC45px_neigh_kmeans_10']]

# Reset the index of data if CellID is currently the index
data_reset = data.reset_index()

# Merge data with the mapping DataFrame using CellID as the key
merged_data = data.merge(mapping_df, on=['CellID', 'core_imageid'], how='left')

# Set CellID back as the index if needed
data = merged_data.set_index('CellID')

In [9]:
rename_dict = {'RCN1': ['2'],
               'RCN2': ['1'],
               'RCN3': ['4'],
               'RCN4': ['6'],
               'RCN5': ['7'],
               'RCN6': ['0'],
               'RCN7': ['3'],
               'RCN8': ['5'],
               'RCN9': ['8'],
               'RCN10': ['9']
                }

replace_dict= {v: k for k, values in rename_dict.items() for v in values}
data['SC45px_neigh_kmeans_10'] = data['SC45px_neigh_kmeans_10'].astype(str)
data['RCNs'] = data['SC45px_neigh_kmeans_10'].replace(replace_dict)

In [10]:
data.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_RCNs.csv')